In [1]:
from textwrap import dedent
from typing import List

from pydantic import BaseModel, Field

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, PromptTemplate
from langchain_core.prompts.image import ImagePromptTemplate
from langchain_ollama import ChatOllama


model = ChatOllama(model='deepseek-v4-pro:cloud', name='main', temperature=0)


class SlideBase(BaseModel):
    slide_image: str = Field(..., description='Detailed English prompt for image generation')
    slide_content: List[str] = Field(..., description='Bullet points shown on the image')
    speaker_notes: str = Field(..., description='Lecture narration for TTS')


class LessonBlueprint(BaseModel):
    slides: List[SlideBase] = Field(
        ...,
        description='Full sequence of slides (exactly 4 slides)'
    )


In [2]:
def build_standard_chat_prompt_template(kwargs):
    """Build a ChatPromptTemplate from system and human message configs.

    Supports both text-only and multimodal (text + image) prompts.

    Args:
        kwargs: Dictionary with optional 'system' and 'human' keys.
            Each value is a dict with 'template' (required) and optional
            'type' ('text' or 'image') and 'input_variables'.

    Returns:
        A configured ChatPromptTemplate ready for use with an LLM.
    """
    messages = []

    if 'system' in kwargs:
        content = kwargs.get('system')

        # allow list of prompts for multimodal
        if isinstance(content, list):
            prompts = [PromptTemplate(**c) for c in content]
        else:
            prompts = [PromptTemplate(**content)]

        message = SystemMessagePromptTemplate(prompt=prompts)
        messages.append(message)

    if 'human' in kwargs:
        content = kwargs.get('human')

        # allow list of prompts for multimodal
        if isinstance(content, list):
            prompts = []
            for c in content:
                if c.get("type") == "image":
                    prompts.append(ImagePromptTemplate(**c))
                else:
                    prompts.append(PromptTemplate(**c))
        else:
            if content.get("type") == "image":
                prompts = [ImagePromptTemplate(**content)]
            else:
                prompts = [PromptTemplate(**content)]

        message = HumanMessagePromptTemplate(prompt=prompts)
        messages.append(message)

    chat_prompt_template = ChatPromptTemplate.from_messages(messages)

    return chat_prompt_template

In [3]:
SYSTEM_PROMPT = dedent('''
# Goal
根據用戶提供的教學需求，產出恰好四頁投影片的完整內容，包含投影片文字與配套的圖像生成 prompt，使下游可直接產出結構化教案與對應視覺素材。

# Role
資深教學設計師兼視覺設計顧問。你擅長將教學主題拆解為四頁循序漸進的投影片，並為每頁撰寫適合圖像生成模型（如 DALL-E、GPT-image）使用的詳細英文視覺 prompt。

# Inputs
- <INPUT>：用戶提供的教學主題、目標受眾、核心知識點或任何課程需求。

# Rules
1. 必須產出恰好四頁投影片，結構為：引入頁（Hook）→ 核心知識頁（Core Concept）→ 展開／案例頁（Elaboration）→ 總結頁（Wrap-up）。
2. 每頁投影片必須包含以下三個欄位：
   - slide_image：完整、詳細的英文圖像生成 prompt（約 3–5 句），描述視覺風格、配色、構圖、文字區域與內容元素。必須使用英文，以便圖像生成模型理解。
   - slide_content：投影片上實際顯示的 bullet points（中文），每頁 3–6 點，簡潔有力。
   - speaker_notes：該頁的教學旁白／講稿（中文），口語化、可朗讀，適合 TTS 生成。使用第一人稱（「我們」、「讓我們」）。
3. slide_image prompt 必須具體描述：
   - 整體視覺風格（如：modern corporate training slide, clean minimalist design, flat illustration）
   - 配色方案（主色、輔助色）
   - 版面佈局（標題區、內容區、視覺元素區）
   - 圖中應出現的具體元素（圖示、圖表、人物場景等）
   - 文字排版要求（字體風格、對齊方式）
4. slide_content 必須與 slide_image 中描述的視覺內容一致，不可出現投影片上未顯示的資訊。
5. speaker_notes 必須比 slide_content 更詳細，補充解釋、過渡語句或互動提問，幫助講師講解。
6. 四頁之間要有邏輯遞進：前一頁的 speaker_notes 末尾應自然引導到下一頁主題。

# Constraints
- slide_image 必須是英文，其餘欄位必須是中文。
- 不可在 slide_content 中使用完整長句，保持 bullet point 格式。
- 不可產出 markdown 程式碼區塊或額外說明文字，僅輸出 JSON。
- 所有字串必須正確 JSON 轉義（如反斜線雙寫）。

# Output
嚴格遵循 format_instructions 指定的 JSON Schema。
輸出必須是一個 LessonBlueprint，其中 slides 陣列恰好包含 4 個 SlideBase 物件。
''')


In [4]:
output_parser = PydanticOutputParser(pydantic_object=LessonBlueprint)
format_instructions = output_parser.get_format_instructions()

input_ = {
        "system": {"template": SYSTEM_PROMPT,
                   "partial_variables": {'format_instructions': format_instructions}},
        "human": {
            "template": "<INPUT>: {input}",
        }
    }

pipeline = build_standard_chat_prompt_template(input_) | model | output_parser


In [5]:
output = pipeline.invoke({"input": "Azur Lane 中的 Sakura Empire 的人物服裝設計"})

In [7]:
# Save pipeline output to JSON for slide_preview.html
import json
with open("output.json", "w", encoding="utf-8") as f:
    json.dump(output.model_dump() if hasattr(output, "model_dump") else output, f, ensure_ascii=False, indent=2)
print("Saved to poc/output.json")


Saved to poc/output.json


In [8]:
# ================================================================
# Layout Generation Pipeline — 第二階段：將單一 SlideBase 轉為 HTML 版面 + 圖片比例
# ================================================================


class SlideLayout(BaseModel):
    """Single slide visual layout design."""
    slide_html: str = Field(
        ...,
        description="Complete HTML <div> string for the slide (16:9, inline CSS only). Must include title zone, bullet zone, and dashed image-placeholder zone."
    )
    image_ratio: str = Field(
        ...,
        description="Recommended image aspect ratio for embedding: 16:9 (full-slide background), 4:3 (content illustration), or 1:1 (icon/avatar)."
    )
    design_rationale: str = Field(
        ...,
        description="Brief Chinese explanation of why this layout and ratio were chosen for THIS slide."
    )


In [12]:
LAYOUT_SYSTEM_PROMPT = dedent('''
# Goal
根據單一投影片的內容（slide_content + slide_image），設計可直接渲染的 HTML 版面，並決定該頁圖片的最佳嵌入比例。

# Role
前端視覺設計師。你擅長將教學內容轉換為 16:9 比例的 HTML 投影片版面，並根據內容類型判斷最適合的圖片比例。

# Inputs
- <SLIDE_CONTENT>：該頁投影片的中文 bullet points（3–6 點）。
- <SLIDE_IMAGE_PROMPT>：該頁投影片的英文圖像生成 prompt（描述視覺風格、配色、構圖等）。
- <SLIDE_NUMBER>：該頁的頁碼（1–4）。

# Rules
1. 只為「這一頁」產出單一 SlideLayout，不可產出多頁或陣列。
2. slide_html 規格：
   - 最外層必須是單一 <div>，使用內聯 CSS（style 屬性），確保 aspect-ratio: 16 / 9。
   - 版面必須包含三個區塊：標題區（大字）、文字內容區（<ul>/<li>）、圖片預留區（虛線邊框 dashed border）。
   - 文字內容必須與 SLIDE_CONTENT 一致，不可新增或刪減 bullet points。
   - 圖片預留區必須標註「圖片預留區：由 slide_image 生成後嵌入」並顯示建議比例。
   - 配色必須參考 SLIDE_IMAGE_PROMPT 中描述的視覺風格與主色。
   - 所有樣式必須寫在 style 屬性中，不可引用外部 CSS。
3. image_ratio 選擇規則：
   - 16:9：當 SLIDE_IMAGE_PROMPT 描述的是全版面背景圖、場景圖或大視覺元素時選用。
   - 4:3：當 SLIDE_IMAGE_PROMPT 描述的是內容配圖、示意圖、流程圖時選用。
   - 1:1：當 SLIDE_IMAGE_PROMPT 描述的是圖示、頭像、小插圖時選用。
   - 預設為 16:9。
4. design_rationale 必須針對該頁內容簡要說明：為什麼選這個 layout 結構？為什麼選這個 image_ratio？

# Constraints
- slide_html 必須是有效的 HTML（單一 root div），不可包含 markdown 代碼區塊標記。
- HTML 中的中文內容必須正確，不可出現未轉義的特殊字元。
- 所有字串必須正確 JSON 轉義（如反斜線雙寫、HTML 標籤中的 < 與 > 需正確處理）。
- 不可產出 markdown 程式碼區塊或額外說明文字，僅輸出 JSON。

# Output

format_instructions: {format_instructions}

嚴格遵循 format_instructions 指定的 JSON Schema。
輸出必須是一個 SlideLayout 物件（非陣列）。
''')


In [13]:
# Layout pipeline — 接收單一 Slide 資料，產出 SlideLayout

layout_parser = PydanticOutputParser(pydantic_object=SlideLayout)
layout_format_instructions = layout_parser.get_format_instructions()

layout_input_ = {
        "system": {"template": LAYOUT_SYSTEM_PROMPT,
                   "partial_variables": {'format_instructions': layout_format_instructions}},
        "human": {
            "template": "Slide number: {slide_number}\nSlide content: {slide_content}\nSlide image prompt: {slide_image_prompt}",
        }
    }

layout_pipeline = build_standard_chat_prompt_template(layout_input_) | model | layout_parser


In [14]:
slide_content = output.slides[0].slide_content
slide_image_prompt = output.slides[0].slide_image


page_1 = layout_pipeline.invoke({"slide_number": 1, "slide_content": output.slides[0].slide_content, "slide_image_prompt": slide_image_prompt})

In [15]:
page_1.slide_html

"<div style='aspect-ratio:16/9;width:100%;background:linear-gradient(135deg,#3d0c0c 0%,#0d0d0d 100%);color:#ffffff;display:flex;flex-direction:column;padding:28px 32px;box-sizing:border-box;font-family:Arial,sans-serif;position:relative;overflow:hidden;border:1px solid #d4af37;'><div style='border-left:6px solid #d4af37;padding-left:16px;margin-bottom:18px;'><h1 style='margin:0;font-size:34px;font-weight:700;letter-spacing:1px;color:#ffffff;'>Sakura Empire：日本陣營的艦船擬人化</h1><div style='margin-top:8px;width:72px;height:2px;background:#d4af37;'></div></div><div style='display:flex;flex:1;gap:28px;align-items:center;'><div style='flex:1.2;display:flex;flex-direction:column;justify-content:center;'><ul style='margin:0;padding-left:22px;list-style:none;'><li style='margin-bottom:14px;font-size:20px;line-height:1.5;color:#f5e6c8;position:relative;padding-left:18px;'><span style='position:absolute;left:0;top:10px;width:8px;height:8px;background:#d4af37;border-radius:50%;'></span>服裝融合傳統和風與軍武元素</l

In [ ]:
# Example: per-slide layout generation (concurrent-ready)

# Stage 1: Generate content blueprint (all slides at once)
# blueprint = pipeline.invoke({"input": "你的教學需求..."})

# Stage 2: Generate layout per slide (can be parallelized later)
# layouts = []
# for idx, slide in enumerate(blueprint.slides, start=1):
#     layout = layout_pipeline.invoke({
#         "slide_number": str(idx),
#         "slide_content": "\n".join(slide.slide_content),
#         "slide_image_prompt": slide.slide_image
#     })
#     layouts.append(layout)
#     print(f"Slide {idx}: ratio={layout.image_ratio}, html_len={len(layout.slide_html)}")

# Future: wrap in ThreadPoolExecutor or asyncio for concurrent calls
# from concurrent.futures import ThreadPoolExecutor
# def render_one(args):
#     return layout_pipeline.invoke(args)
# with ThreadPoolExecutor(max_workers=4) as pool:
#     layouts = list(pool.map(render_one, slide_args_list))


In [ ]:
output